In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import string
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print('Տվյալների բեռնում...')
df = pd.read_csv('bbc-text (1).csv')
print(f'Ընդհանուր հոդվածների քանակը՝ {len(df)}')



STOP_WORDS = {
    'i',
    'me',
    'my',
    'myself',
    'we',
    'our',
    'ours',
    'ourselves',
    'you',
    'your',
    'yours',
    'yourself',
    'yourselves',
    'he',
    'him',
    'his',
    'himself',
    'she',
    'her',
    'hers',
    'herself',
    'it',
    'its',
    'itself',
    'they',
    'them',
    'their',
    'theirs',
    'themselves',
    'what',
    'which',
    'who',
    'whom',
    'this',
    'that',
    'these',
    'those',
    'am',
    'is',
    'are',
    'was',
    'were',
    'be',
    'been',
    'being',
    'have',
    'has',
    'had',
    'having',
    'do',
    'does',
    'did',
    'doing',
    'a',
    'an',
    'the',
    'and',
    'but',
    'if',
    'or',
    'because',
    'as',
    'until',
    'while',
    'of',
    'at',
    'by',
    'for',
    'with',
    'about',
    'against',
    'between',
    'into',
    'through',
    'during',
    'before',
    'after',
    'above',
    'below',
    'to',
    'from',
    'up',
    'down',
    'in',
    'out',
    'on',
    'off',
    'over',
    'under',
    'again',
    'further',
    'then',
    'once',
    'here',
    'there',
    'when',
    'where',
    'why',
    'how',
    'all',
    'any',
    'both',
    'each',
    'few',
    'more',
    'most',
    'other',
    'some',
    'such',
    'no',
    'nor',
    'not',
    'only',
    'own',
    'same',
    'so',
    'than',
    'too',
    'very',
    's',
    't',
    'can',
    'will',
    'just',
    'don',
    'should',
    'now',
}


def preprocess_text(text):
  if not isinstance(text, str):
    return []
  text = text.lower()
  text = text.translate(str.maketrans('', '', string.punctuation))
  return [w for w in text.split() if w not in STOP_WORDS]


print('Տեքստերի մշակում (Tokenization & Cleaning)...')
df['tokens'] = df['text'].apply(preprocess_text)



def build_vocabulary(tokenized_docs, max_features=2500):
  word_counts = {}
  for doc in tokenized_docs:
    for word in doc:
      word_counts[word] = word_counts.get(word, 0) + 1
  sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
  vocab = [word for word, count in sorted_words[:max_features]]
  return {word: i for i, word in enumerate(vocab)}


print('Կառուցվում է բառարանը (Vocabulary)...')
vocab_dict = build_vocabulary(df['tokens'], max_features=2500)
vocab_size = len(vocab_dict)
print(f'Բառարանի չափը (V) = {vocab_size}')


print('Ստեղծվում են BoW վեկտորները...')
bow_matrix = np.zeros((len(df), vocab_size), dtype=int)
for i, tokens in enumerate(df['tokens']):
  for word in tokens:
    if word in vocab_dict:
      bow_matrix[i, vocab_dict[word]] += 1



print('Հաշվարկվում են TF-IDF արժեքները...')

doc_lengths = np.array([len(doc) if len(doc) > 0 else 1 for doc in df['tokens']])
tf_matrix = bow_matrix / doc_lengths[:, np.newaxis]


doc_freq = np.sum(bow_matrix > 0, axis=0)
N = len(df)
idf_vector = np.log(N / (doc_freq + 1)) + 1


tfidf_matrix = tf_matrix * idf_vector
print(f'TF-IDF մատրիցը պատրաստ է։ Չափը՝ {tfidf_matrix.shape}')


categories = df['category'].unique()
target_cat = categories[0]
print(
    f"\n--- Վերլուծություն '{target_cat}' կատեգորիայի համար (Թոփ 10 բառեր) ---"
)

cat_indices = df[df['category'] == target_cat].index
mean_tfidf = np.mean(tfidf_matrix[cat_indices], axis=0)

idx_to_word = {i: w for w, i in vocab_dict.items()}
top_10_idx = mean_tfidf.argsort()[::-1][:10]

for rank, idx in enumerate(top_10_idx, 1):
  print(f"{rank}. Բառ: '{idx_to_word[idx]}' | Միջին TF-IDF: {mean_tfidf[idx]:.4f}")

print('\n Ամեն ինչ հաջողությամբ ավարտվեց։')